In [1]:
import os
import sys
import pandas as pd
from typing import Tuple
import math

In [2]:
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [3]:
from baseline.turbulence_benchmark.utility.turbulence_log_functions import TurbulenceLogHelper

In [4]:
def compare_multiple_code_generation_logs(res_dir: str, filter: Tuple[str] = (), anti_filter: Tuple[str] = ()):
    csv_logs = [f for f in os.listdir(res_dir) if (
        os.path.isfile(os.path.join(res_dir, f)) and 
        f.endswith(".csv") and 
        all(sub in f for sub in filter)) and
        all(sub not in f for sub in anti_filter)
        ]

    log_file_names = [csv_file_name.replace('.csv', '') for csv_file_name in csv_logs]
    df_row_names = TurbulenceLogHelper.obtain_row_names(file_names = log_file_names)

    results_df = pd.DataFrame(
        index = df_row_names,
        columns= ["Code Inconsistency Score"]
        )

    for _ in range(len(df_row_names)):
        log1_file_name = csv_logs.pop()
        for log2_file_name in csv_logs:
            log1_file_path = os.path.join(res_dir, log1_file_name)
            log2_file_path = os.path.join(res_dir, log2_file_name)

            log1 = pd.read_csv(log1_file_path)
            log2 = pd.read_csv(log2_file_path) 
            print(log1_file_name, log2_file_name)
            loghelper = TurbulenceLogHelper()
            score_dict = loghelper.obtain_mucoco_code_inconsistency_score(log1=log1, log2=log2)

            log1_inconsistencies = score_dict["log1_inconsistencies"]
            log2_inconsistencies = score_dict["log2_inconsistencies"]
            total_correct = score_dict["total_correct"]
            total_inconsistencies = log1_inconsistencies + log2_inconsistencies

            entry_row_name = f"{log1_file_name.replace('.csv', '')} - {log2_file_name.replace('.csv', '')}"
            results_df.loc[entry_row_name, "Code Inconsistency Score"] = f"{total_inconsistencies}/{total_correct} = {round( number= (total_inconsistencies/total_correct)*100, ndigits=2)}%"
    
    return results_df

In [ ]:
def question_inconsistency(res_dir: str, filter: Tuple[str] = (), anti_filter: Tuple[str] = ()):
    csv_logs = [f for f in os.listdir(res_dir) if (
        os.path.isfile(os.path.join(res_dir, f)) and 
        f.endswith(".csv") and 
        all(sub in f for sub in filter)) and
        all(sub not in f for sub in anti_filter)
        ]
    
    helper = TurbulenceLogHelper()

    log_file_names = [csv_file_name.replace('.csv', '') for csv_file_name in csv_logs]

    results_df = pd.DataFrame(
        index = log_file_names,
        columns= ["Question Inconsistency Score"]
        )
    
    for file_name in csv_logs:
        log_file_path = os.path.join(res_dir, file_name)
        entry_log = pd.read_csv(log_file_path)
        inconsistency_dict = helper.obtain_question_inconsistency_count(entry_log)
        inconsistency_qn_count = inconsistency_dict['inconsistent_qn_count']
        total_tasks = inconsistency_dict["total_tasks"]

        results_df.loc[file_name.replace('.csv', '')] = f"{inconsistency_qn_count}/ {total_tasks}"

    return results_df

In [6]:
res_dir = proj_dir + "/results/code_generation/gpt-4o"

res = compare_multiple_code_generation_logs(res_dir=res_dir, anti_filter=("BigCodeBench", "HumanEval"))
res2 = question_inconsistency(res_dir=res_dir, anti_filter=("BigCodeBench", "HumanEval"))

print(res)
print(res2)

Turbulence_zero_shot_random.csv Turbulence_zero_shot_no_mutation.csv
Starting comparison of 52 tasks...

=== COMPARISON SUMMARY ===
Total tasks processed: 52
Both succeeded: 383
Both failed: 0
Comparable tasks (atleast one succeeded): 433
  - Log1 failed, Log2 succeeded: 29
  - Log1 succeeded, Log2 failed: 21
Total inconsistencies: 50/433
Turbulence_zero_shot_random.csv Turbulence_zero_shot_sequential.csv
Starting comparison of 52 tasks...

=== COMPARISON SUMMARY ===
Total tasks processed: 52
Both succeeded: 404
Both failed: 0
Comparable tasks (atleast one succeeded): 437
  - Log1 failed, Log2 succeeded: 33
  - Log1 succeeded, Log2 failed: 0
Total inconsistencies: 33/437
Turbulence_zero_shot_sequential.csv Turbulence_zero_shot_no_mutation.csv
Starting comparison of 52 tasks...

=== COMPARISON SUMMARY ===
Total tasks processed: 52
Both succeeded: 410
Both failed: 0
Comparable tasks (atleast one succeeded): 439
  - Log1 failed, Log2 succeeded: 2
  - Log1 succeeded, Log2 failed: 27
Total 